#### 1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

structure = StructType([
    StructField('customer_id', IntegerType(), True),
    StructField('customer_name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('phone', StringType(), True),
])

data = [
    (1, 'John Doe', 'john.doe@example.com', '123-456-7890'),
    (2, 'Jane Smith', 'jane.smith@example.com', '987-654-3210'),
    (3, 'Alice Johnson', 'alice.johnson@example.com', '555-123-4567'),
]

df = spark.createDataFrame(data, structure)
display(df)
df.write.mode("overwrite").format("delta").saveAsTable("cyntexa_dev.sales.customers")

In [0]:
dataNew = [
    (3, 'Alice Brown', 'alice.johnson@example.com', '555-123-4567'),
    (4, 'Bob Brown', 'bob.brown@example.com', '111-222-3333')
]

dfNew = spark.createDataFrame(dataNew, structure)
display(dfNew)
dfNew.write.mode("overwrite").format("delta").saveAsTable("cyntexa_dev.sales.customers_new")

In [0]:
%sql
merge into cyntexa_dev.sales.customers as t
using cyntexa_dev.sales.customers_new as s
on t.customer_id = s.customer_id
when matched then update set *
when not matched then insert *

In [0]:
%sql
select * from cyntexa_dev.sales.customers

#### 2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity Catalog permissions.

In [0]:
%sql
grant select on table cyntexa_dev.sales.customers to group cyntexa_dev

#### 3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms, what a DBU is billing for.

In [0]:
%sql
-- Check if we have any billing data at all
SELECT 
  MIN(usage_date) as earliest_date,
  MAX(usage_date) as latest_date,
  COUNT(*) as total_records,
  SUM(CASE WHEN usage_unit = 'DBUs' THEN usage_quantity ELSE 0 END) as total_dbus
FROM system.billing.usage

In [0]:
%sql
-- See what types of usage we have and recent SKUs
SELECT 
  usage_date,
  sku_name,
  usage_unit,
  SUM(usage_quantity) as total_usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 7 DAYS
GROUP BY usage_date, sku_name, usage_unit
ORDER BY usage_date DESC, total_usage DESC
LIMIT 20

In [0]:
%sql
-- DBU consumption for All-Purpose Serverless Compute (what you've been using)
SELECT 
  usage_date,
  sku_name,
  ROUND(SUM(usage_quantity), 4) as dbus_consumed,
  usage_unit
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 7 DAYS
  AND sku_name LIKE '%ALL_PURPOSE_SERVERLESS%'
GROUP BY usage_date, sku_name, usage_unit
ORDER BY usage_date DESC

## What is a DBU (Databricks Unit)?

**In plain terms:** A DBU is Databricks' billing unit that measures how much computing power you use over time.

### What you're actually paying for:

1. **Compute Time** - The amount of time your cluster or serverless compute is running and processing data
   - Each minute your compute runs consumes DBUs
   - More powerful machines consume DBUs faster

2. **Processing Power** - The size and type of the compute resource
   - A larger cluster (more cores, more memory) consumes more DBUs per minute
   - Different workload types have different DBU rates:
     - All-Purpose compute (interactive notebooks like this one)
     - Jobs compute (scheduled tasks)
     - SQL compute (warehouses for queries)
     - Model inference (serving ML models)

3. **Premium Features** - Enhanced capabilities
   - Serverless compute (instant start, auto-scaling)
   - Security features
   - Advanced collaboration tools

### Your Usage Example:

Looking at your **PREMIUM_ALL_PURPOSE_SERVERLESS_COMPUTE** usage:
- **Today (Aug 27):** 2.15 DBUs - Running this notebook and doing light data work
- **Aug 26:** 10.01 DBUs - More intensive work that day
- **Aug 25:** 10.54 DBUs - Similar workload

### Think of it like electricity:
- DBUs are like kilowatt-hours for your electricity bill
- Just as electricity billing = power used × time, DBUs = compute power × time
- A bigger compute resource is like running a large appliance - it uses more units per hour
- The serverless option is like having a smart grid that scales up/down automatically, so you only pay for what you actively use